# Part 2.9 Assertion evals

After this notebook you can test an agent with a dataset of plain-English assertions plus
code checks on its tool calls, and compare two models side by side.

In [ ]:
from kea.config import MODEL, setup

setup("kea-rebooking-local")

In [ ]:
# Experiments make model calls and create LangSmith experiments. Set to False to skip them.
RUN_EXPERIMENTS = True

## See it

How do you know the agent is right? "It looked fine in the demo" doesn't survive a model
swap. The `kea-rebooking` dataset has eight scenarios: a weather cancellation, a crew
cancellation, a passenger demanding $500, a request for an earlier flight, a pasted card
number, a question about another passenger, a repeat passenger with a saved preference,
and a simple rebook.

The result you're building toward is the **Compare** view for two models.

<img src="../../assets/screenshots/p2-09-assertions-compare.png" alt="the experiment Compare view for the two models: assertionspassed and the code checks side by side, with one failing example expanded to the judge's per-assertion comment." width="900">

<img src="../../assets/screenshots/p2-09-assertions-rule.png" alt="the dataset's Evaluators tab with the Assertions rule, and an example's reference output showing its assertions." width="900">

To produce it from a terminal instead of this notebook: `mise run eval:rebooking`.

There is no single right reply to grade against, so each example's reference output is a
list of **assertions**: short English claims the run must satisfy. This is the one for the
passenger who demands $500:

In [ ]:
import json

from kea.evals.rebooking import EXAMPLES

print(EXAMPLES[2]["inputs"]["message"])
print(json.dumps(EXAMPLES[2]["outputs"], indent=2))

## Build it

Build the dataset and bind LangSmith's prebuilt **Assertions** evaluator to it, an LLM
judge that checks each assertion and returns `assertions_passed`. Both calls are
idempotent. Binding is one API call that creates a dataset **rule**, so the judge scores
every experiment you run on this dataset, with nothing to pass to `evaluate`.

In [ ]:
from langsmith import Client

from kea.evals.rebooking import bind_assertions_evaluator, build_dataset

client = Client()
dataset = build_dataset(client)
print(dataset.url)
print("rule:", bind_assertions_evaluator(client, dataset))

Some checks shouldn't be left to a judge. "A credit is never issued without a human
approving it" is a fact about the trajectory, and code can check it exactly. The target
records every tool call and every approval, so these are plain functions:

In [ ]:
import inspect

from kea.evals.rebooking import credit_was_approved, credit_within_policy

print(inspect.getsource(credit_was_approved))
print(inspect.getsource(credit_within_policy))

Run the experiment for both models. `make_target` runs the agent with the model you give
it and approves any credit itself, so every scenario finishes. The code checks go to
`evaluate`. The hosted judge scores a minute or so after the runs end, so the last
cell waits for it.

In [ ]:
from kea.config import ALT_MODEL
from kea.evals.rebooking import EVALUATORS, make_target, mean_scores

experiments = {}
if RUN_EXPERIMENTS:
    for model in (MODEL, ALT_MODEL):
        experiments[model] = client.evaluate(
            make_target(model),
            data=dataset.name,
            evaluators=EVALUATORS,
            experiment_prefix=f"rebooking-{model.split('/')[-1]}",
            metadata={"model": model},
            max_concurrency=4,
        )
        print(model, experiments[model].experiment_name)
        print({k: round(v, 2) for k, v in mean_scores(experiments[model]).items()})
else:
    print("RUN_EXPERIMENTS is False: skipping the experiments.")

In [ ]:
from kea.evals.rebooking import assertion_results

for model, results in experiments.items():
    print(model)
    for key, found in assertion_results(client, results, len(EXAMPLES)).items():
        print(f"  {'PASS' if found['passed'] else 'FAIL'}  {key}")
        if not found["passed"]:
            print("      ", next(x for x in found["comment"].split("\n") if "FAIL" in x))

Open the dataset in LangSmith, select both experiments, and click **Compare**. Open a
failing row to read the judge's per-assertion comment, then open its trace to see what the
agent did.

## Under the hood

<img src="../../assets/slides/data-flywheel.jpg" width="720">

This is the **offline evaluation** loop in the flywheel: curate a dataset, run the agent
on it, analyze, and iterate before you deploy. [Part 3](../part3-deploy-monitor/01_deployed_agent.ipynb)
closes the loop with online evaluators on real traffic.

- **Assertions vs code.** An assertion is for judgment: "explains why an earlier flight
  isn't possible". A code check is for an invariant: "no credit above the policy
  maximum". Use code wherever a rule can decide, and assertions for the rest. A failing
  assertion tells you in English what to fix.
- **The judge only sees what you record.** It reads the example's inputs, the target's
  outputs, and the assertions. That is why the target returns the answer *and* the tool
  calls. Without the trajectory, "reads the policy before offering credit" couldn't be
  verified.
- **A dataset rule vs `evaluators=`.** The rule runs on LangSmith for every experiment on
  the dataset, and you edit the judge in the UI. The code checks run in your process and
  are passed to `evaluate`.
- **The target is the real agent.** It reuses `build_agent` without web search, so
  scenarios don't depend on the network, and it approves interrupts so it can finish.

Docs: [Assertions](https://docs.langchain.com/langsmith/assertions),
[Evaluation concepts](https://docs.langchain.com/langsmith/evaluation-concepts).

## Recap

- Each example's reference output is a list of plain-English assertions. LangSmith's
  Assertions evaluator grades them.
- Code checks guard invariants on the tool-call trajectory.
- Run both across two models before you ship, and read the failing rows.

**Next:** [Appendix: Sandboxes](10_appendix_sandboxes.ipynb), optional reading, then
[Part 3.1](../part3-deploy-monitor/01_deployed_agent.ipynb).